## What makes Transformers Transformers?
Multi-head, scaled dot-product attention.

Multi-head? Yes, just run the same thing (attention) multiple times over the same thing (input) with different weights (heads).

Scaled dot-product? Yes, as the name suggests-- here's your scaled dot-product (matmul): $$\frac{1}{\sqrt{d_k}}Q\cdot K^T$$

Attention? A weighting function, really, just a fancier name. Take the result above, make it into probabilities (read: normalize; done using softmax) and multiply that resulting thing (matrix of probability vectors from query and key matrics) by a "value" matrix: $$\text{probabilities}=\text{softmax}(\frac{1}{\sqrt{d_k}}Q \cdot K^T)$$ $$\text{attention}=\text{probabilities} \cdot V$$

That's it.

In [1]:
import torch
import torch.nn as nn
import math

def sdp(queries, keys):
    dk = queries.shape[-1]
    sdp = 1/torch.sqrt(torch.tensor(dk)) * queries.matmul(keys.T)
    return sdp

In [27]:
dk = 10
queries = torch.ones(2, dk)
keys = torch.ones(2,dk)

sdp(queries, keys), queries

(tensor([[3.1623, 3.1623],
         [3.1623, 3.1623]]),
 tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]))

Without the scaling by $\frac{1}{\sqrt{d_k}}$, resulting values in first matrix are 10, not "3.163".

In [28]:
def sdp_attn(queries, keys, values):
    pass

Where do we get the "values" matrix from? Also, somewhere here, we need to plug the input.. probably in this V matrix, right?

## Ah! Q, K and V are *not* weights!
$$Q=X \cdot W_Q$$
$$K=X \cdot W_K$$
$$V=X \cdot W_V$$

In [37]:
from torch.nn.functional import softmax

hidden_dim = 16
input_dim = 2 # this is the vocab(?) -> no, embedding dim.. that's what that was for..
dk = 6

X = torch.ones(input_dim, hidden_dim)
W_Q = torch.randn((hidden_dim, dk))
W_K = torch.randn((hidden_dim, dk))
W_V = torch.randn((hidden_dim, dk))


def sdp_attn(X, Q_proj, K_proj, V_proj):
    Q = torch.matmul(X, Q_proj)
    K = torch.matmul(X, K_proj)
    
    sdp = 1/math.sqrt(dk) * torch.matmul(Q, K.T)
    
    probs = softmax(sdp, dim=0)
    
    V = torch.matmul(X, V_proj)
    
    attn = torch.matmul(probs, V)
    
    return attn

In [38]:
sdp_attn(X, W_Q, W_K, W_V)

tensor([[-1.5090,  1.6827, -2.5519, -1.7468, -0.1020, -3.2838],
        [-1.5090,  1.6827, -2.5519, -1.7468, -0.1020, -3.2838]])

That's.. it.. I guess?

Attention matrix has dimensions of: input_dim x dk

input_dim is NOT the vocabulary size; it's the embedding dimension i.e. when we make our nn.Embedding(vocab_size, embedding_dim), that's the dimension it uses for a given vocab-sized vector (the key for the embedding dict, the value being the embedding vector)

So the attention matrix is of dimension "embedding_dim x d_k"

## Back to the "multi-heads"
Multi-head just means we have different $W_Q$, $W_K$, $W_V$ for each head.

Q: how are they recombined for next layer?

A: they're concatenated

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads):
        self.heads = nn.ModuleList([AttentionHead])